<a href="https://colab.research.google.com/github/N-umwali/SchoolCare-Screening-Priority/blob/main/student_screening.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **SchoolCare: Student Screening Model**
Survey C dataset is used to train the machine learning model to prioritise students for counsellor-led depression screening using 34 non-symptom predictors. This notebook reproduces the completed analysis with the original split, model settings and decision rules.

### **Step 1: Prepare Survey C**
The first cell locates the project folders, loads the original Excel file, and checks the expected 2,905 records and 56 columns.

**Import important libraries**

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

**Load Survey C
Load the dataset and create output folders.**

In [8]:
ROOT = Path("/content/SchoolCare-Screening-Priority")
RAW = ROOT / "data/raw"
DATA = ROOT / "data/processed"
REPORTS = ROOT / "reports"
FIGURES = REPORTS / "figures"
ARTIFACTS = ROOT / "model_artifacts"

for folder in (RAW, DATA, REPORTS, FIGURES, ARTIFACTS):
    folder.mkdir(parents=True, exist_ok=True)

source = RAW / "survey_c_reidentified.xlsx"

if not source.is_file():
    uploaded = files.upload()
    if source.name not in uploaded:
        raise FileNotFoundError("Upload survey_c_reidentified.xlsx.")
    source.write_bytes(uploaded[source.name])

raw = pd.read_excel(source)
assert raw.shape == (2905, 56), f"Unexpected data shape: {raw.shape}"
print("Original data shape:", raw.shape)

Original data shape: (2905, 56)


Survey C contains `2,905` records and `56` columns, as expected. Eligibility checks follow in the next cell.

**Identify eligible students**

A recorded school allows every student from that school to stay in the same evaluation partition. All eight PHQ responses are required to calculate the outcome.

### **Remove Missing School Codes**
Treat blank school codes as missing and retain students with a school code.

In [9]:
phq_cols = [f"PHQ_{i}" for i in range(1, 9)]

# Treat blank school codes as missing.
school = raw["school_name"].astype("string").str.strip().replace("", pd.NA)
missing_school = int(school.isna().sum())

with_school = raw.loc[school.notna()].copy()
with_school["school_name"] = school.loc[school.notna()]

### **Define the Outcome**
Retain complete PHQ-8 responses, validate scores, and label totals ≥10 as elevated symptoms. Confirm cohort counts.

In [10]:
# A complete PHQ-8 score needs all eight answers.
phq = with_school[phq_cols].apply(pd.to_numeric, errors="coerce")
incomplete_phq = int(phq.isna().any(axis=1).sum())

eligible = with_school.loc[phq.notna().all(axis=1)].copy()
valid_phq = phq.loc[eligible.index].isin([0, 1, 2, 3]).all().all()
assert valid_phq, "An eligible student has an invalid PHQ answer."

eligible["PHQ_sum"] = phq.loc[eligible.index].sum(axis=1).astype(int)
eligible["outcome"] = (eligible["PHQ_sum"] >= 10).astype(int)

assert missing_school == 654
assert incomplete_phq == 452
assert len(eligible) == 1799
assert eligible["school_name"].nunique() == 32

print(f"Original records: {len(raw)}")
print(f"Missing school: {missing_school}")
print(f"Incomplete PHQ-8 after school exclusion: {incomplete_phq}")
print(f"Eligible students: {len(eligible)}")
print(f"Schools: {eligible['school_name'].nunique()}")
print(f"Elevated PHQ-8 symptoms: {eligible['outcome'].sum()} "
      f"({eligible['outcome'].mean():.2%})")

Original records: 2905
Missing school: 654
Incomplete PHQ-8 after school exclusion: 452
Eligible students: 1799
Schools: 32
Elevated PHQ-8 symptoms: 535 (29.74%)


Of the `2,905` records, `654` had no school identifier. Among the remaining records, 452 had an incomplete PHQ-8 questionnaire.

**Check survey answer codes**

The approved predictors and attention-check answer are checked against their permitted codes. Undefined answers become missing without removing otherwise eligible students.

In [11]:
support = ["MSPSS_3", "MSPSS_4", "MSPSS_8", "MSPSS_11"]
strain = [f"FSS_{i}" for i in range(1, 5)]
aspirations = [f"AI_{i}" for i in range(1, 16)]

other = [
    "Age", "Gender", "Form", "Religion", "Surviving_Parents",
    "Parents_Dead", "Fathers_Education", "Mothers_Education",
    "Co_Curricular", "Sports", "Percieved_Academic_Abilities"
]
predictors = support + strain + aspirations + other
assert len(predictors) == len(set(predictors)) == 34

valid_codes = {
    **{col: set(range(1, 8)) for col in support + aspirations},
    **{col: set(range(1, 6)) for col in strain},
    "Gender": {1, 2},
    "Form": {1, 2, 3, 4},
    "Religion": set(range(1, 8)),
    "Surviving_Parents": {0, 1, 2},
    "Parents_Dead": {1, 2, 3, 4},
    "Fathers_Education": {1, 2, 3, 4},
    "Mothers_Education": {1, 2, 3, 4},
    "Co_Curricular": {1, 2, 3},
    "Sports": {1, 2},
    "Percieved_Academic_Abilities": {1, 2, 3, 4, 5},
    "GAD_Check": {0, 1, 2, 3},
}

### **Measure the undefined Religion code before changing it.**

In [22]:
# Recreate eligible students before cleaning.
phq_cols = [f"PHQ_{i}" for i in range(1, 9)]
phq = with_school[phq_cols].apply(pd.to_numeric, errors="coerce")

eligible = with_school.loc[phq.notna().all(axis=1)].copy()

assert phq.loc[eligible.index].isin([0, 1, 2, 3]).all().all(), (
    "An eligible student has an invalid PHQ answer."
)

eligible["PHQ_sum"] = phq.loc[eligible.index].sum(axis=1).astype(int)
eligible["outcome"] = (eligible["PHQ_sum"] >= 10).astype(int)

In [23]:
# Count undefined Religion codes before cleaning.
religion_code_8 = int(
    pd.to_numeric(eligible["Religion"], errors="coerce").eq(8).sum()
)

quality_rows = []

for col in predictors + ["GAD_Check"]:
    original = eligible[col]
    values = pd.to_numeric(original, errors="coerce")

    invalid_numeric = original.notna() & values.isna()

    # Age must be a positive whole number.
    invalid_age = (
        (col == "Age")
        & values.notna()
        & ((values <= 0) | (values % 1 != 0))
    )

    # Other columns must contain valid codebook values.
    invalid_code = (
        (col != "Age")
        & values.notna()
        & ~values.isin(valid_codes.get(col, []))
    )

    invalid = invalid_numeric | invalid_age | invalid_code
    eligible[col] = values.mask(invalid)

    quality_rows.append({
        "column": col,
        "invalid_codes": int(invalid.sum()),
        "missing_after_cleaning": int(eligible[col].isna().sum()),
        "missing_percent": round(100 * eligible[col].isna().mean(), 2),
    })

quality = pd.DataFrame(quality_rows)

assert religion_code_8 == 80, (
    f"Expected 80 Religion=8 values, found {religion_code_8}."
)
assert quality.set_index("column").loc["Religion", "invalid_codes"] == 80
assert quality.loc[
    quality["column"] != "Religion", "invalid_codes"
].sum() == 0

unusual_ages = (
    eligible["Age"].notna()
    & ~eligible["Age"].between(13, 21)
)

print("Undefined Religion=8 values set to missing:", religion_code_8)
print(
    "Ages outside the observed 13–21 range to review:",
    int(unusual_ages.sum())
)
display(quality)

Undefined Religion=8 values set to missing: 80
Ages outside the observed 13–21 range to review: 0


,column,invalid_codes,missing_after_cleaning,missing_percent
0,MSPSS_3,0,56,3.11
1,MSPSS_4,0,76,4.22
2,MSPSS_8,0,65,3.61
3,MSPSS_11,0,60,3.34
4,FSS_1,0,65,3.61
5,FSS_2,0,65,3.61
6,FSS_3,0,78,4.34
7,FSS_4,0,61,3.39
8,AI_1,0,50,2.78
9,AI_2,0,51,2.83


The codebook defines Religion codes `1–7`, but `80` eligible students had code `8`. Those answers were marked as missing while keeping the students in the dataset.

### **Separate inputs and save prepared data**

Only the `34` approved non-symptom predictors enter the model-input table. The school identifier is reserved for grouping and GAD_Check for the secondary sensitivity analysis.

In [31]:
# GAD_Check is kept only for the later sensitivity analysis.
failed_check = int(
    (eligible["GAD_Check"].notna() & eligible["GAD_Check"].ne(0)).sum()
)
blank_check = int(eligible["GAD_Check"].isna().sum())

assert failed_check == 188
assert blank_check == 181

X = eligible[predictors].copy()
y = eligible["outcome"].copy()
groups = eligible["school_name"].copy()

assert X.shape == (1799, 34)
assert X.columns.tolist() == predictors
assert X.index.equals(y.index) and X.index.equals(groups.index)
print("Approved model inputs:", X.shape)


Approved model inputs: (1799, 34)


In [32]:
prepared = eligible[
    predictors + ["outcome", "school_name", "GAD_Check"]
].copy()

exclusions = pd.DataFrame([
    ("Original records", len(raw)),
    ("Excluded: missing school", missing_school),
    ("Excluded next: incomplete PHQ-8", incomplete_phq),
    ("Eligible students", len(eligible)),
    ("Schools", eligible["school_name"].nunique()),
], columns=["measure", "count"])

prepared.to_csv(DATA / "survey_c_eligible.csv", index=False)
quality.to_csv(REPORTS / "step1_quality_report.csv", index=False)
exclusions.to_csv(REPORTS / "step1_exclusion_summary.csv", index=False)

print("Nonzero, nonblank GAD_Check:", failed_check)
print("Blank GAD_Check:", blank_check)
print("Saved: survey_c_eligible.csv")
print("Saved: step1_quality_report.csv")
print("Saved: step1_exclusion_summary.csv")

Nonzero, nonblank GAD_Check: 188
Blank GAD_Check: 181
Saved: survey_c_eligible.csv
Saved: step1_quality_report.csv
Saved: step1_exclusion_summary.csv


The model-input table has `1,799` students and exactly` 34` approved non-symptom predictors. The school identifier is reserved for splitting students by school; GAD_Check is retained only for a later sensitivity check.